In [0]:
# CELL 1: CONFIGURATION & SECURITY (SILVER LAYER)
# -----------------------------------------------
# Define storage details
storage_account_name = "dlpeopleanalytics2026"
container_name = "medallion-data"

# Authenticate securely using Azure Key Vault
storage_account_key = dbutils.secrets.get(scope="kv-secrets", key="storage-account-key")

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.blob.core.windows.net",
    storage_account_key
)

In [0]:
# CELL 2: READ BRONZE LAYER DATA (INDEPENDENT INCREMENTAL BATCHES)
# ----------------------------------------------------------------
from pyspark.sql.functions import col, max as spark_max

# Define base path for the bronze layer
bronze_base_path = f"wasbs://{container_name}@{storage_account_name}.blob.core.windows.net/bronze"

# Native Spark function to avoid precision loss in microseconds
def get_latest_batch(table_name):
    df_temp = spark.read.format("delta").load(f"{bronze_base_path}/{table_name}")
    
    # 1. Search for the maximum timestamp directly in a 1-row DataFrame
    max_time_df = df_temp.select(spark_max("ingestion_timestamp").alias("max_time"))
    
    # 2. Cross join that value against the full table and filter natively
    return df_temp.crossJoin(max_time_df) \
                  .filter(col("ingestion_timestamp") == col("max_time")) \
                  .drop("max_time")

# Read tables applying the secure function
df_headcount_bronze = get_latest_batch("headcount")
df_roles_bronze = get_latest_batch("dim_role")
df_salaries_bronze = get_latest_batch("reference_salaries")
df_survey_bronze = get_latest_batch("climate_survey")

print("Read complete. Each table filtered its own timestamp natively in Spark.")

Read complete. Each table filtered its own timestamp natively in Spark.


In [0]:
# CELL 3: DATA CLEANING & DEDUPLICATION (BATCH SNAPSHOT)
# ------------------------------------------------------
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, desc

# 1. Drop redundant market salary column from dim_role
df_roles_cleaned = df_roles_bronze.drop("average_market_salary")

# 2. Define Window specifications for deduplication
# Partitioning by ID AND snapshot_date to ensure we keep the latest record PER MONTH
window_roles = Window.partitionBy("role_id", "snapshot_date").orderBy(desc("ingestion_timestamp"))
window_headcount = Window.partitionBy("employee_id", "snapshot_date").orderBy(desc("ingestion_timestamp"))
# FIXED: Updated 'id_rol' to 'role_id' to match the English schema
window_salaries = Window.partitionBy("role_id", "snapshot_date").orderBy(desc("ingestion_timestamp"))
window_survey = Window.partitionBy("employee_id", "snapshot_date").orderBy(desc("ingestion_timestamp"))

# 3. Apply the Window functions to keep only the latest valid record in this batch per snapshot
df_roles_silver_draft = df_roles_cleaned.withColumn("rn", row_number().over(window_roles)).filter(col("rn") == 1).drop("rn")
df_headcount_silver_draft = df_headcount_bronze.withColumn("rn", row_number().over(window_headcount)).filter(col("rn") == 1).drop("rn")
df_salaries_silver_draft = df_salaries_bronze.withColumn("rn", row_number().over(window_salaries)).filter(col("rn") == 1).drop("rn")
df_survey_silver_draft = df_survey_bronze.withColumn("rn", row_number().over(window_survey)).filter(col("rn") == 1).drop("rn")

print("All DataFrames successfully deduplicated and partitioned by snapshot_date.")

All DataFrames successfully deduplicated and partitioned by snapshot_date.


In [0]:
# CELL 4: DATA QUALITY, FORMATTING & QUARANTINE ROUTING
# -----------------------------------------------------
from pyspark.sql.functions import col, trim, initcap, lit, to_date, regexp_replace
from pyspark.sql.types import DoubleType, DateType

# 1. FORMAT STANDARDIZATION & DATA TYPE CASTING (FIXED FOR SYNAPSE COMPATIBILITY)
df_roles_std = df_roles_silver_draft \
    .withColumn("department", initcap(trim(col("department")))) \
    .withColumn("role_name", initcap(trim(col("role_name")))) \
    .withColumn("snapshot_date", to_date(col("snapshot_date"), "yyyy-MM-dd").cast(DateType()))

df_salaries_std = df_salaries_silver_draft \
    .withColumn("average_market_salary", regexp_replace(col("average_market_salary").cast("string"), ",", ".").cast(DoubleType())) \
    .withColumn("min_market_salary", regexp_replace(col("min_market_salary").cast("string"), ",", ".").cast(DoubleType())) \
    .withColumn("max_market_salary", regexp_replace(col("max_market_salary").cast("string"), ",", ".").cast(DoubleType())) \
    .withColumn("snapshot_date", to_date(col("snapshot_date"), "yyyy-MM-dd").cast(DateType()))

df_headcount_std = df_headcount_silver_draft \
    .withColumn("hire_date", to_date(col("hire_date"), "yyyy-MM-dd").cast(DateType())) \
    .withColumn("termination_date", to_date(col("termination_date"), "yyyy-MM-dd").cast(DateType())) \
    .withColumn("snapshot_date", to_date(col("snapshot_date"), "yyyy-MM-dd").cast(DateType())) \
    .withColumn("current_salary", regexp_replace(col("current_salary").cast("string"), ",", ".").cast(DoubleType()))

df_survey_std = df_survey_silver_draft \
    .withColumn("snapshot_date", to_date(col("snapshot_date"), "yyyy-MM-dd").cast(DateType()))

# 2. INTRINSIC QUALITY CHECKS (NULL-Safe Patch Applied)
hc_invalid_cond = (col("current_salary") <= 0) | \
                  col("employee_id").isNull() | \
                  col("role_id").isNull() | \
                  col("hire_date").isNull() | \
                  (col("termination_date").isNotNull() & (col("termination_date") < col("hire_date")))
                  # ^ SINGLE LINE ADDED: Previously, the "controlled chaos" of hire_date was null
                  # (5% of the rows, intentionally in the generator) and was not filtered here and
                  # was even filtered through to Gold. Now it's quarantined along with the rest.

survey_invalid_cond = (col("general_satisfaction_score") < 1) | \
                      (col("general_satisfaction_score") > 5)

df_hc_valid_step1 = df_headcount_std.filter(~hc_invalid_cond)
df_hc_quarantine_intrinsic = df_headcount_std.filter(hc_invalid_cond).withColumn("error_reason", lit("Intrinsic Quality: Salary <= 0, Null IDs, Null hire_date, or Invalid Dates"))

df_survey_valid_step1 = df_survey_std.filter(~survey_invalid_cond)
df_survey_quarantine_intrinsic = df_survey_std.filter(survey_invalid_cond).withColumn("error_reason", lit("Intrinsic Quality: Satisfaction score out of bounds"))

# 3. REFERENTIAL INTEGRITY CHECKS (Keys + Snapshot Date)
df_hc_quarantine_ref = df_hc_valid_step1.join(df_roles_std, ["role_id", "snapshot_date"], "left_anti").withColumn("error_reason", lit("Referential Integrity: role_id not found in dim_role for this period"))
df_headcount_final = df_hc_valid_step1.join(df_roles_std, ["role_id", "snapshot_date"], "left_semi")

df_survey_quarantine_ref = df_survey_valid_step1.join(df_headcount_final, ["employee_id", "snapshot_date"], "left_anti").withColumn("error_reason", lit("Referential Integrity: employee_id not found in Headcount for this period"))
df_survey_final = df_survey_valid_step1.join(df_headcount_final, ["employee_id", "snapshot_date"], "left_semi")

df_roles_quarantine_ref = df_roles_std.join(df_salaries_std, ["role_id", "snapshot_date"], "left_anti").withColumn("error_reason", lit("Referential Integrity: role_id not found in market salaries API for this period"))
df_roles_final = df_roles_std.join(df_salaries_std, ["role_id", "snapshot_date"], "left_semi")

df_salaries_final = df_salaries_std

# 4. CONSOLIDATE QUARANTINE TABLES
df_headcount_quarantine_final = df_hc_quarantine_intrinsic.unionByName(df_hc_quarantine_ref, allowMissingColumns=True)
df_survey_quarantine_final = df_survey_quarantine_intrinsic.unionByName(df_survey_quarantine_ref, allowMissingColumns=True)
df_roles_quarantine_final = df_roles_quarantine_ref

print("Data Quality process completed. Dates and Doubles forced for Synapse compatibility.")

Data Quality process completed. Dates and Doubles forced for Synapse compatibility.


In [0]:
# CELL 5: LOAD TO SILVER LAYER (PERIODIC SNAPSHOTS & QUARANTINE)
# --------------------------------------------------------------

# Define base paths for the Silver layer
silver_base_path = f"wasbs://{container_name}@{storage_account_name}.blob.core.windows.net/silver"
quarantine_base_path = f"{silver_base_path}/quarantine"

# Write VALID DataFrames to Silver (APPEND mode to create the Periodic Snapshots history)
df_roles_final.write.format("delta").mode("append").save(f"{silver_base_path}/dim_role")
df_headcount_final.write.format("delta").mode("append").save(f"{silver_base_path}/headcount")
df_salaries_final.write.format("delta").mode("append").save(f"{silver_base_path}/reference_salaries")
df_survey_final.write.format("delta").mode("append").save(f"{silver_base_path}/climate_survey")

# Write QUARANTINE DataFrames to Silver (APPEND mode to keep a historical log of errors)
df_roles_quarantine_final.write.format("delta").mode("append").save(f"{quarantine_base_path}/dim_role_err")
df_headcount_quarantine_final.write.format("delta").mode("append").save(f"{quarantine_base_path}/headcount_err")
df_survey_quarantine_final.write.format("delta").mode("append").save(f"{quarantine_base_path}/climate_survey_err")

print("Silver Layer materialization complete: Historical Snapshots and Quarantine logs successfully appended.")

Silver Layer materialization complete: Historical Snapshots and Quarantine logs successfully appended.


In [0]:
# CELL 6: DATA QUALITY AUDIT REPORT
# ---------------------------------
print("=========================================")
print("       SILVER LAYER AUDIT REPORT         ")
print("=========================================")

roles_valid = df_roles_final.count()
roles_quar = df_roles_quarantine_final.count()
print(f"Dim Role           | Valid: {roles_valid:5} | Quarantined: {roles_quar:5}")

hc_valid = df_headcount_final.count()
hc_quar = df_headcount_quarantine_final.count()
print(f"Headcount          | Valid: {hc_valid:5} | Quarantined: {hc_quar:5}")

salaries_valid = df_salaries_final.count()
print(f"Reference Salaries | Valid: {salaries_valid:5} | Quarantined:     0")

survey_valid = df_survey_final.count()
survey_quar = df_survey_quarantine_final.count()
print(f"Climate Survey     | Valid: {survey_valid:5} | Quarantined: {survey_quar:5}")

print("=========================================")

       SILVER LAYER AUDIT REPORT         
Dim Role           | Valid:  1290 | Quarantined:     0
Headcount          | Valid: 71088 | Quarantined:  7696
Reference Salaries | Valid:  1290 | Quarantined:     0
Climate Survey     | Valid:  9444 | Quarantined:  1053
